# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library and Croissant schema. The dataset describes 77 cancer survivors with second primary colorectal cancer, providing clinical, pathological, and molecular variables that support further research and analysis.

### Dataset Source
The dataset is defined via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}\n")
# Optionally pretty-print other fields
# pprint.pprint(metadata.to_json())

## 2. Data Overview
Review available record sets (`RecordSet`), their `@id`s, and available fields and columns (by `@id`).

In [ ]:
# List available record sets and their IDs
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '')}")

# For each record set, print available fields (with their @id) and columns
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name', '')})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        # fields may be a reference or Embedded, check accordingly
        if isinstance(f, dict):
            print(f"   - @id: {f.get('@id', '<unknown>')} | name: {f.get('name', '')}")
        elif isinstance(f, str):
            print(f"   - @id: {f}")
    print("  Columns:")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    for c in columns:
        if isinstance(c, dict):
            print(f"   - @id: {c.get('@id', '<unknown>')} | name: {c.get('name', '')}")
        elif isinstance(c, str):
            print(f"   - @id: {c}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Set record set ids (from the data overview section above):
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # records() yields dicts with field @id as keys
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded {len(df)} records for Record Set @id='{record_set_id}'. Columns (Field @ids):")
        print(df.columns.tolist())
        display(df.head())
    else:
        print(f"\nRecord Set @id='{record_set_id}' has no records.")

# Select the main clinical record set if known, else pick the one with most rows
if len(dataframes) > 0:
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"Using Record Set: {main_record_set_id}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by a numeric field, normalizing a column, or grouping data by a key variable. You may adjust field @ids as needed, referring explicitly to all columns by their `@id`.

In [ ]:
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Exploring DataFrame for Record Set @id: {main_record_set_id}\n")
    # Try to identify a numeric field automatically
    numeric_candidates = []
    for col in df.columns:
        # Heuristic: if dtype is numeric, or column name contains 'age', 'interval', or 'count'
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_candidates.append(col)
            name_lower = col.lower()
            if any(w in name_lower for w in ['age', 'interval', 'count', 'year', 'months', 'days']):
                numeric_candidates.append(col)
        except Exception:
            continue

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field for EDA: {numeric_field_id}\n")
    else:
        print("No numeric field could be automatically detected.")
        numeric_field_id = df.columns[0]  # fallback

    # Try to convert the field to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Simple threshold for filtering demonstration
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a grouping variable: look for categorical columns
    group_field_id = None
    for c in df.columns:
        if c != numeric_field_id and (df[c].dtype == object or df[c].dtype.name == 'category'):
            # Pick a field with low cardinality
            if df[c].nunique() <= 5:
                group_field_id = c
                break
    if group_field_id:
        print(f"\nGrouping records by field {group_field_id}:")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped_df)
    else:
        print("No clear group field detected for aggregation.")
else:
    print("No main record set DataFrame loaded.")

## 5. Visualization
Visualize distributions or relationships between fields using the loaded DataFrame. All fields are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    if numeric_field_id in df:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of field '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
    if group_field_id and numeric_field_id in df:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated the use of the `mlcroissant` library to load, inspect, and perform exploratory analysis on the FAIR² cancer survivor colorectal cancer dataset. By working directly with the Croissant schema, we can robustly reference columns, record sets, and variables by their global `@id`, making our data workflow portable and standards-compliant. Further analysis can continue based on the extracted DataFrame, with all metadata traceable via `@id` mappings.